<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 05. BERT

## BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

- **원 논문:** [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- **저자 / 발표:** Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova · NAACL 2019 (2018)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 80/10/10 MLM corruption과 bidirectional encoder를 재현한다.

**축소하거나 생략한 부분:** BERT Base/Large와 BooksCorpus+Wikipedia 대신 1 encoder block을 쓰고, 300개 train 행과 60개 held-out 행을 서로 다른 seed로 손상한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Task #1: MLM and 80% mask/10% random/10% unchanged | Task 1 | train/evaluation별 selected labels와 정확한 80/10/10 corruption |
| Fig. 1 and §3 Fig. 2: bidirectional attention and three embeddings | Task 2 explicit unmasked Q/K/V attention | [B,H,T,T] positive attention weights |
| §3.1 Task #1: MLM objective | Task 3 | loss and masked-token accuracy |

## 핵심 아이디어와 수식

$$L_{MLM}=-\sum_{i\in\mathcal{M}}\log p(x_i\mid x_{\setminus\mathcal{M}})$$

**기호 해설:** M은 선택된 position, x_i는 원 token, 양쪽 context가 조건으로 사용된다.

**코드 대응:** Task 1이 80/10/10 corruption, Task 2가 mask 없는 Q/K/V 양방향 encoder, Task 3이 MLM CE를 구현한다.

**입력과 출력 shape:** train corruption [300,4], held-out corruption [60,4]이며, 10-row probe는 hidden [10,4,24]와 logits [10,4,V]를 만든다.

**포트폴리오 구현 범위:** NSP와 대규모 corpus는 제외하지만 masking 정책과 비인과적 self-attention은 보존한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Task #1: MLM and 80% mask/10% random/10% unchanged
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** train/evaluation별 selected labels와 정확한 80/10/10 corruption

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class BERTConfig:
    model_dim: int = 24
    heads: int = 4
    feedforward_dim: int = 48
    learning_rate: float = 0.025
    steps: int = 32


def apply_mlm_corruption(
    tokens,
    mask_id,
    random_token_ids,
    generator,
):
    corrupted = tokens.clone()
    labels = torch.full_like(corrupted, -100)
    batch_size, sequence_length = tokens.shape
    rows = torch.arange(batch_size, device=tokens.device)
    positions = torch.randint(
        sequence_length,
        (batch_size,),
        generator=generator,
    ).to(tokens.device)
    labels[rows, positions] = corrupted[rows, positions]

    corruption_order = torch.randperm(
        batch_size,
        generator=generator,
    ).to(tokens.device)
    mask_count = round(0.8 * batch_size)
    random_count = round(0.1 * batch_size)
    mask_rows = corruption_order[:mask_count]
    random_rows = corruption_order[mask_count : mask_count + random_count]
    corrupted[mask_rows, positions[mask_rows]] = mask_id
    random_offsets = torch.randint(
        len(random_token_ids),
        (random_count,),
        generator=generator,
    ).to(tokens.device)
    corrupted[random_rows, positions[random_rows]] = random_token_ids[random_offsets]
    return corrupted, labels


config = BERTConfig()
split_generator = torch.Generator().manual_seed(31)
sentence_order = torch.randperm(
    len(sentence_ids),
    generator=split_generator,
).to(sentence_ids.device)
train_sentence_index = sentence_order[:300]
test_sentence_index = sentence_order[300:]
assert not bool(torch.isin(train_sentence_index, test_sentence_index).any())
random_token_ids = torch.tensor(
    [stoi[word] for word in words],
    device=sentence_ids.device,
)
train_corrupted, train_mlm_labels = apply_mlm_corruption(
    sentence_ids[train_sentence_index],
    MASK,
    random_token_ids,
    torch.Generator().manual_seed(41),
)
test_corrupted, test_mlm_labels = apply_mlm_corruption(
    sentence_ids[test_sentence_index],
    MASK,
    random_token_ids,
    torch.Generator().manual_seed(42),
)
train_selected = train_mlm_labels != -100
test_selected = test_mlm_labels != -100
assert int(train_selected.sum()) == 300
assert int(test_selected.sum()) == 60
assert int((train_corrupted[train_selected] == MASK).sum()) == 240
assert int((test_corrupted[test_selected] == MASK).sum()) == 48

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Fig. 1 and §3 Fig. 2: bidirectional attention and three embeddings
- **노트북에서 구현할 부분:** Task 2 explicit unmasked Q/K/V attention
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** [B,H,T,T] positive attention weights

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class BidirectionalSelfAttention(nn.Module):
    """모든 token pair의 scaled dot-product score를 계산한다."""

    def __init__(self, config):
        super().__init__()
        if config.model_dim % config.heads != 0:
            raise ValueError("model_dim must be divisible by heads")
        self.heads = config.heads
        self.head_dim = config.model_dim // config.heads
        self.query_projection = nn.Linear(
            config.model_dim,
            config.model_dim,
        )
        self.key_projection = nn.Linear(
            config.model_dim,
            config.model_dim,
        )
        self.value_projection = nn.Linear(
            config.model_dim,
            config.model_dim,
        )
        self.output_projection = nn.Linear(
            config.model_dim,
            config.model_dim,
        )

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.reshape(
            batch,
            length,
            self.heads,
            self.head_dim,
        )
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.reshape(
            batch,
            length,
            self.heads * self.head_dim,
        )

    def forward(self, hidden):
        queries = self.split_heads(self.query_projection(hidden))
        keys = self.split_heads(self.key_projection(hidden))
        values = self.split_heads(self.value_projection(hidden))
        scale = self.head_dim**-0.5
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
        weights = scores.softmax(dim=-1)
        context = torch.matmul(weights, values)
        output = self.output_projection(self.merge_heads(context))
        return output, weights


class BERTMini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, config.model_dim)
        self.position_embedding = nn.Embedding(4, config.model_dim)
        self.segment_embedding = nn.Embedding(2, config.model_dim)
        self.attention = BidirectionalSelfAttention(config)
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.GELU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.mlm_head = nn.Linear(config.model_dim, vocab_size)

    def embed(self, token_ids):
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)
        segments = torch.zeros_like(token_ids)
        return (
            self.token_embedding(token_ids)
            + self.position_embedding(positions)[None]
            + self.segment_embedding(segments)
        )

    def forward(self, token_ids):
        embedded = self.embed(token_ids)
        attended, _ = self.attention(embedded)
        hidden = self.norm1(embedded + attended)
        hidden = self.norm2(hidden + self.feedforward(hidden))
        return self.mlm_head(hidden)


model = BERTMini(VOCAB_SIZE, config).to(DEVICE)
corruption_probe = train_corrupted[:10]
assert model.embed(corruption_probe).shape == (10, 4, 24)
assert model(corruption_probe).shape == (10, 4, VOCAB_SIZE)
_, bidirectional_weights = model.attention(model.embed(corruption_probe))
assert bidirectional_weights.shape == (10, 4, 4, 4)
assert torch.all(bidirectional_weights > 0)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Task #1: MLM objective
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** loss and masked-token accuracy

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def masked_language_model_loss(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
        ignore_index=-100,
    )


def train_bert_step(model, optimizer, corrupted, labels):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = masked_language_model_loss(model(corrupted), labels)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_mlm(model, corrupted, labels):
    model.eval()
    selected = labels != -100
    with torch.no_grad():
        predictions = model(corrupted)[selected].argmax(dim=-1)
    return float((predictions == labels[selected]).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_bert_step(
            model,
            optimizer,
            train_corrupted,
            train_mlm_labels,
        )
    )
heldout_accuracy = evaluate_mlm(
    model,
    test_corrupted,
    test_mlm_labels,
)
assert min(losses[3:]) < losses[0]
assert heldout_accuracy >= 0.5
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"BERT held-out MLM accuracy={heldout_accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> BERT Base/Large와 BooksCorpus+Wikipedia 대신 1 encoder block을 쓰고, 300개 train 행과 60개 held-out 행을 서로 다른 seed로 손상한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.